In [ ]:
import pyspark

In [ ]:
pyspark

Подключение к релиационным базам полностью поддерживается

# 1. JDBC: Работа с реляционными базами данных в Spark

## Базовый паттерн подключения

Классический вызов spark.read.jdbc() создает один JDBC connection и читает всю таблицу в один partition:

In [12]:
# Базовое чтение - МЕДЛЕННО (1 connection, 1 partition)
df = spark.read.jdbc(url, table, properties)

In [ ]:
Проблема: Не использует распределенную природу Spark, читает всё в одном executor'е.

In [ ]:
# Способы чтения

# Вариант 1: Простая таблица
.option("dbtable", "employees")

# Вариант 2: Схема + таблица
.option("dbtable", "hr.employees")

# Вариант 3: Подзапрос (полезно для фильтрации/джойнов)
.option("dbtable", "(SELECT * FROM employees WHERE dept_id = 10) AS emp")

# Вариант 4: Использование query (нельзя с partitionColumn!)
.option("query", "SELECT name, salary FROM employees WHERE hire_date > '2020-01-01'")

## Параллельное чтение

Параметр        | Обязательность | Описание
----------------|----------------|--------------------------------------------------------
partitionColumn | Да             | Числовой/дата столбец для разбиения
lowerBound      | Да             | Минимальное значение столбца
upperBound      | Да             | Максимальное значение столбца
numPartitions   | Да             | Количество параллельных соединений

In [ ]:
df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:...") \
    .option("dbtable", "table") \
    .option("partitionColumn", "id")     # ← параллелизм
    .option("lowerBound", "1")           # ← параллелизм  
    .option("upperBound", "1000000")     # ← параллелизм
    .option("numPartitions", "4")        # ← параллелизм
    .option("fetchsize", "10000")        # ← оптимизация
    .option("pushDownPredicate", "true") # ← фильтрация в БД
    .load()

**Важно:** partitionColumn, lowerBound, upperBound должны указываться вместе.

Как работает: Spark делит диапазон [lowerBound, upperBound] на numPartitions равных частей и создает отдельный SQL запрос для каждой части с WHERE условием.

## Параметры чтения

| Параметр | Значение по умолч. | Что делает | Когда менять |
|----------|-------------------|------------|--------------|
| `fetchsize` | 0 | Сколько строк забирать за один запрос | Oracle (по умолч. 10) → ставьте 1000-10000 |
| `queryTimeout` | 0 | Таймаут запроса в секундах | При долгих запросах увеличьте |
| `pushDownPredicate` | true | "Проталкивать" фильтры в SQL | Если БД быстрее фильтрует |
| `pushDownAggregate` | true | "Проталкивать" агрегации в SQL | Если БД быстрее агрегирует |
| `numPartitions` | - | Максимум параллельных соединений | Баланс: скорость vs нагрузка на БД |

## Баланс между скоростью и давлением на источник

- Слишком много partitions: Перегрузка БД (Thundering Herd проблема)

- Слишком мало partitions: Не используем возможности Spark

Золотая середина:

- numPartitions = 4-16 для большинства случаев

- Не превышать 70% от max_connections БД

Практические советы узнать параметры из postgres

In [ ]:
# Шаг 1: Узнать лимиты БД
SELECT name, setting 
FROM pg_settings 
WHERE name LIKE '%connections%';
-- postgres: max_connections = 100

    
    
# Шаг 2: Узнать размер данных
SELECT COUNT(*) as total_rows, 
       MAX(id) - MIN(id) as id_range
FROM large_table;




# Шаг 3: Рассчитать оптимальный размер чанка
optimal_chunk_size = 100_000  # 100K строк на партицию - хороший баланс



# Шаг 4: Применить формулу
total_rows = 10_000_000
max_connections = 100
executor_cores = 4
num_executors = 3



partitions_by_data = total_rows // optimal_chunk_size  # 100
partitions_by_db = max_connections * 0.7  # 70
partitions_by_spark = executor_cores * num_executors  # 12



optimal = min(100, 70, 12)  # = 12 partitions

## 1. Перегрузка БД  (Thundering Herd)

In [ ]:
# ❌ ПЛОХО: 100 параллельных соединений
.option("numPartitions", 100)

# ✅ ХОРОШО: 8-16 соединений
.option("numPartitions", 8)

## 2. Неравномерные партиции (Data Skew)

Возникает, если `partitionColumn` имеет неравномерное распределение:

* 90% данных в одной партиции

* Остальные партиции почти пустые

Решение: Выбирать равномерно распределенный столбец (первичный ключ).

## 3. Long-running transactions

В PostgreSQL/MySQL долгие транзакции чтения блокируют autovacuum.

In [ ]:
# ПРОБЛЕМА: Большие транзакции в READ COMMITTED
df = spark.read.jdbc(
    ...,
    dbtable="table_with_100M_rows",  # Чтение 100M строк
    isolationLevel="READ_COMMITTED"   # Каждая строка видит снапшот
)
# → В PostgreSQL создается long-running snapshot
# → Autovacuum не может чистить старые версии строк
# → БД раздувается, замедляется

# ПРОБЛЕМА 2: REPEATABLE READ в MySQL
.option("isolationLevel", "REPEATABLE_READ")
# → MySQL создает read view на ВСЕ время транзакции
# → UNDO лог растет бесконечно

In [ ]:
# РЕШЕНИЕ 1: Разбить на чанки по времени
def read_in_chunks(start_date, end_date, chunk_days=7):
    current = start_date
    while current < end_date:
        chunk_end = min(current + timedelta(days=chunk_days), end_date)
        
        df_chunk = spark.read.jdbc(
            ...,
            dbtable=f"""(SELECT * FROM table 
                      WHERE created_at BETWEEN '{current}' AND '{chunk_end}') t""",
            numPartitions=4  # Мало партиций для маленького чанка
        )
        
        # Обработка chunk
        process(df_chunk)
        
        current = chunk_end
        print(f"Processed chunk to {chunk_end}")
        time.sleep(5)  # Даем БД передышку

In [ ]:
# РЕШЕНИЕ 2: Легкие уровни изоляции
df = spark.read \
    .option("isolationLevel", "READ_UNCOMMITTED")  # Самый легкий
    # ИЛИ
    .option("isolationLevel", "READ_COMMITTED")    # Баланс

In [ ]:
# РЕШЕНИЕ 3: Уменьшить fetchsize
.option("fetchsize", 1000)  # Маленькие пачки
.option("queryTimeout", 300)  # Убить запрос через 5 мин`

# 1.2 Запись через JDBC

## Режимы записи

In [ ]:
# Добавление данных
.mode("append")

# Перезапись (опасно!)
.mode("overwrite")

**Batch insert** - самый важный параметр

In [ ]:
# ❌ МЕДЛЕННО: 1 запрос на строку (по умолчанию)
.option("batchsize", 1)

# ✅ БЫСТРО: 10000-50000 строк за запрос
.option("batchsize", 50000)

Оптимальные значения:

- batchsize = 10000 для строк с большими текстами

- batchsize = 50000 для обычных строк

Примеры:

In [ ]:
# Медленная запись (1 запрос на строку)
df.write.jdbc(url, table, mode="append", properties=props)

# Быстрая запись (batch-вставки)
df.write \
  .mode("append") \
  .format("jdbc") \
  .option("url", url) \
  .option("dbtable", table) \
  .option("batchsize", 50000)  # ← САМЫЙ ВАЖНЫЙ ПАРАМЕТР
  .save()

In [ ]:
# Запись с оптимизациями
(df.write
  .mode("overwrite")
  .format("jdbc")
  .option("url", "jdbc:postgresql://localhost/db")
  .option("dbtable", "target_table")
  .option("user", "user")
  .option("password", "pass")
  .option("batchsize", "20000")
  .option("truncate", "true")  # Если поддерживается
  .save())

## Isolation level и блокировки

.option("isolationLevel", "READ_COMMITTED")

Уровни изоляции (от быстрого к надежному):

1. `READ_UNCOMMITTED` - быстрее всего

2. `READ_COMMITTED` - баланс (рекомендуется)

3. `REPEATABLE_READ` - для точности

4. `SERIALIZABLE` - медленно, но надежно

TRUNCATE vs DROP для overwrite

In [ ]:
# Быстрее, сохраняет индексы (не все БД поддерживают)
.option("truncate", "true")

## Потенциальные проблемы записи

1. **Deadlocks** (взаимные блокировки)

Возникают при параллельной записи в одни и те же таблицы.

Решение: Использовать append + staging таблицы вместо конкурентных overwrite.

2. **Большие вставки и OutOfMemory**

Слишком большой batchsize может привести к нехватке памяти.

Решение: Уменьшить batchsize до 10000-20000.

## Стратегии безопасной записи больших объемов

## 1. Staging Table + Atomic Rename

In [ ]:
# 1. Пишем во временную таблицу
df.write.jdbc(..., table="staging_table", mode="overwrite")

# 2. Атомарный swap на стороне БД
# (выполняется SQL в БД, не в Spark)
swap_sql = """
BEGIN;
DROP TABLE IF EXISTS target_table_old;
ALTER TABLE target_table RENAME TO target_table_old;
ALTER TABLE staging_table RENAME TO target_table;
COMMIT;
"""
spark.read.jdbc(..., dbtable=f"({swap_sql}) t")

## 2. Partitioned Write для огромных данных

In [ ]:
# Разбиваем данные по месяцам и пишем отдельно
for month in months:
    month_data = df.filter(f"month = '{month}'")
    month_data.write.jdbc(..., mode="append")

# 1.3 Практика

## Чтение крупной таблицы без убийства базы

In [ ]:
def safe_read_large_table():
    # 1. Узнаем размер таблицы
    count_query = spark.read.jdbc(
        url, 
        dbtable="(SELECT COUNT(*) as cnt, MAX(id) as max_id FROM table) t",
        numPartitions=1
    )# РЕШЕНИЕ 3: Уменьшить fetchsize
.option("fetchsize", 1000)  # Маленькие пачки
.option("queryTimeout", 300)  # Убить запрос через 5 мин
    
    row_count = count_query.collect()[0]["cnt"]
    
    # 2. Рассчитываем оптимальное число партиций
    if row_count < 1000000:
        num_partitions = 2
    elif row_count < 10000000:
        num_partitions = 4
    else:
        # Не более 16, чтобы не перегрузить БД
        num_partitions = min(16, row_count // 500000)
    
    # 3. Читаем с оптимальными параметрами
    df = spark.read \
        .format("jdbc") \
        .option("url", url) \
        .option("dbtable", "table") \
        .option("partitionColumn", "id") \
        .option("lowerBound", 1) \
        .option("upperBound", count_query.collect()[0]["max_id"]) \
        .option("numPartitions", num_partitions) \
        .option("fetchsize", 10000) \
        .load()
    
    return df

## Массовая выгрузка в staging-таблицу

In [ ]:
def write_to_staging_with_retry(df, max_retries=3):
    """
    Запись с ретраями и промежуточными коммитами
    """
    for attempt in range(max_retries):
        try:
            # 1. Пишем во staging
            df.write \
                .option("batchsize", 20000) \
                .option("isolationLevel", "READ_COMMITTED") \
                .jdbc(url, "staging_table", mode="overwrite")
            
            # 2. Валидируем данные
            staging_count = spark.read.jdbc(url, 
                "(SELECT COUNT(*) as cnt FROM staging_table) t").collect()[0]["cnt"]
            source_count = df.count()
            
            if staging_count == source_count:
                print(f"Запись успешна: {staging_count} строк")
                return True
            else:
                print(f"Расхождение: source={source_count}, staging={staging_count}")
                raise Exception("Data mismatch")
                
        except Exception as e:
            print(f"Попытка {attempt + 1} не удалась: {e}")
            if attempt == max_retries - 1:
                raise
            time.sleep(5 * (attempt + 1))  # Exponential backoff
    
    return False

## Incremental загрузки

In [ ]:
def incremental_load_etl():
    """
    Загрузка данных из продакшен БД в Data Warehouse
    с использованием watermark для отслеживания прогресса
    """
    # Конфигурация
    SOURCE_URL = "jdbc:postgresql://prod-db:5432/production"
    TARGET_URL = "jdbc:postgresql://dw:5432/data_warehouse"
    
    SOURCE_TABLE = "public.orders"
    TARGET_TABLE = "dw.fact_orders"
    WATERMARK_TABLE = "dw.etl_watermarks"
    
    # 1. Получаем последний загруженный ID из DW
    #    (watermark хранится В DW, это метаданные ETL процесса)
    watermark_query = f"""
    SELECT last_loaded_id 
    FROM {WATERMARK_TABLE} 
    WHERE source_table = '{SOURCE_TABLE}' 
      AND target_table = '{TARGET_TABLE}'
    """
    
    watermark_df = spark.read.jdbc(TARGET_URL, dbtable=f"({watermark_query}) t")
    
    if watermark_df.count() > 0:
        last_id = watermark_df.collect()[0]["last_loaded_id"]
        print(f"Найден watermark: last_id = {last_id}")
    else:
        last_id = 0  # Первая загрузка
        print("Watermark не найден, начинаем с начала")
    
    # 2. Загружаем новые данные ИЗ ПРОДАКШЕН БД
    #    Используем параметры для безопасного чтения:
    #    - numPartitions=4: не перегружаем БД
    #    - fetchsize=10000: читаем порциями
    #    - WHERE id > last_id: только новые данные
    #    - AND updated_at > ...: дополнительная защита от старых данных
    new_data = spark.read.jdbc(
        SOURCE_URL,
        dbtable=f"""
        (SELECT * FROM {SOURCE_TABLE} 
         WHERE id > {last_id} 
         AND updated_at > CURRENT_DATE - INTERVAL '30 days') t
        """,
        numPartitions=4,
        fetchsize=10000
    )
    
    if new_data.count() == 0:
        print("Нет новых данных для загрузки")
        return
    
    print(f"Найдено {new_data.count()} новых записей")
    
    # 3. Находим максимальный ID в новых данных
    #    Это будет новый watermark
    new_max_id = new_data.agg({"id": "max"}).collect()[0][0]
    
    # 4. Записываем данные В DW
    #    Используем batchsize=50000 для эффективной вставки
    print(f"Загружается {new_data.count()} записей...")
    new_data.write.jdbc(
        TARGET_URL,
        table=TARGET_TABLE,
        mode="append",
        batchsize=50000
    )
    
    # 5. Обновляем watermark В DW
    #    Используем UPSERT (INSERT ... ON CONFLICT) для атомарности
    update_watermark_sql = f"""
    INSERT INTO {WATERMARK_TABLE} 
        (source_table, target_table, last_loaded_id, loaded_at)
    VALUES 
        ('{SOURCE_TABLE}', '{TARGET_TABLE}', {new_max_id}, CURRENT_TIMESTAMP)
    ON CONFLICT (source_table, target_table) 
    DO UPDATE SET 
        last_loaded_id = EXCLUDED.last_loaded_id,
        loaded_at = EXCLUDED.loaded_at
    """
    
    spark.read.jdbc(
        TARGET_URL,
        dbtable=f"({update_watermark_sql}) t",
        numPartitions=1
    )
    
    print(f"Загрузка завершена. Новый watermark: {new_max_id}")

Что делает этот код:
1. Watermark (закладка): Хранит ID последней загруженной записи

2. Incremental (инкрементально): Загружает только новые данные с момента последнего запуска

3. Safe (безопасно): Не перезаписывает старые данные, только добавляет новые

4. Resumable (возобновляемо): Если упадет - в следующий раз продолжит с того же места

Пример выполнения
```
День 1: Загружены записи с ID 1-1000 → Watermark = 1000
День 2: В source появились ID 1001-1500
       Код видит watermark=1000 → загружает только 1001-1500 → Watermark = 1500
День 3: В source появились ID 1501-1800
       Код видит watermark=1500 → загружает только 1501-1800 → Watermark = 1800
```

## Практический пример (Greenplum при ЧТЕНИИ через JDBC)

**Spark думает что это "просто PostgreSQL"**

In [ ]:

# Spark думает что это "просто PostgreSQL"
df = spark.read.jdbc(
    url="jdbc:postgresql://gp-master:5432/db",  # Подключаемся к MASTER
    table="large_table",
    properties={"user": "...", "password": "..."}
)

Что происходит внутри Greenplum:

- Spark подключается к Master Node

- Master Node получает запрос SELECT * FROM large_table

- Master рассылает запрос всем Segment Nodes

- Каждый Segment выполняет свою часть параллельно

- Master собирает результаты и отдает Spark

- Но для Spark это выглядит как обычный JDBC!

In [ ]:
df.write.jdbc(
    url="jdbc:postgresql://gp-master:5432/db",
    table="target_table",
    mode="append"
)

- Spark пишет данные в Master

- Master распределяет данные по Segment'ам согласно DISTRIBUTED BY ключу

- Каждый Segment пишет в свою часть таблицы

## ПРАКТИЧЕСКИЕ СОВЕТЫ ДЛЯ ПРОДАКШЕНА

Оптимизация больших выгрузок:
1. Всегда используйте staging таблицы для overwrite операций

2. Начинайте с малого: numPartitions=4, batchsize=10000

3. Мониторьте нагрузку на БД во время выполнения

4. Разбивайте огромные таблицы на временные интервалы

5. Используйте incremental загрузки там, где возможно

**Избегайте перегрузки базы:**

In [ ]:
# ❌ ОПАСНЫЕ ПРАКТИКИ:
- numPartitions > 50
- Чтение всей таблицы без фильтров
- Overwrite больших таблиц без staging
- Параллельные запуски одной и той же ETL

# ✅ БЕЗОПАСНЫЕ ПРАКТИКИ:
- numPartitions = 4-16
- WHERE условия для ограничения данных
- Staging таблицы + atomic rename
- Lock/координация между параллельными процессами